In [1]:
# Run this ONCE. Then comment it out so it doesn't reinstall every time.
%pip install earthengine-api geemap geopandas folium geopy scikit-learn matplotlib --quiet


Note: you may need to restart the kernel to use updated packages.


## Step 1 - Imports and authentication

In [ ]:
import ee
import geemap
from datetime import datetime, timedelta
from geopy.geocoders import Nominatim
import json

print("Libraries loaded")

Libraries loaded ✓


In [ ]:
# First time: this opens a browser to log in to your Google account
# After the first time, you can comment this line out
# ee.Authenticate()

# Replace with YOUR project ID from Google Cloud
GEE_PROJECT = "floodsat-497417"

ee.Initialize(project=GEE_PROJECT)
print("Connected to Google Earth Engine")

Connected to Google Earth Engine ✓


## Step 2 - Pick a location

In [ ]:
LOCATION = "Sekinchan, Selangor, Malaysia"   # Prompt location here
RADIUS_KM = 5                                # Prompt radius here

geocoder = Nominatim(user_agent="flood_app")
place = geocoder.geocode(LOCATION)
lat, lon = place.latitude, place.longitude

# Build the Area of Interest (AOI)
center = ee.Geometry.Point([lon, lat])
aoi = center.buffer(RADIUS_KM * 1000).bounds()

print(f"📍 Location: {LOCATION}")
print(f"   Coordinates: ({lat:.4f}, {lon:.4f})")
print(f"   AOI size: {RADIUS_KM*2} × {RADIUS_KM*2} km")

📍 Location: Sekinchan, Selangor, Malaysia
   Coordinates: (3.5064, 101.1027)
   AOI size: 10 × 10 km


## Step 3 - Choose date windows

In [ ]:
today = datetime.utcnow().date()
last_year = today.year - 1

# Current window
NOW_END   = today.strftime("%Y-%m-%d")
NOW_START = (today - timedelta(days=30)).strftime("%Y-%m-%d")

# Flood season last year (Malaysia: Nov–Feb)
FLOOD_START = f"{last_year}-11-01"
FLOOD_END   = f"{last_year+1}-02-28"

# Dry season last year (Malaysia: Jun–Aug)
DRY_START = f"{last_year}-06-01"
DRY_END   = f"{last_year}-08-31"

print(f"Now    : {NOW_START} → {NOW_END}")
print(f"Flood  : {FLOOD_START} → {FLOOD_END}")
print(f"Dry    : {DRY_START} → {DRY_END}  (baseline)")

Now    : 2026-04-25 → 2026-05-25
Flood  : 2025-11-01 → 2026-02-28
Dry    : 2025-06-01 → 2025-08-31  (baseline)


/var/folders/dj/lts6hwzx1gqbq581ntz95lbr0000gn/T/ipykernel_14413/1405994084.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow().date()


## Step 4 - Download Sentinel-1 radar images

In [ ]:
def get_radar(start_date, end_date):
    '''Get a median Sentinel-1 image for the given period.'''
    collection = (ee.ImageCollection("COPERNICUS/S1_GRD")
                  .filterBounds(aoi)
                  .filterDate(start_date, end_date)
                  .filter(ee.Filter.eq("instrumentMode", "IW"))
                  .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
                  .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
                  .select(["VV", "VH"]))
    return collection.median().clip(aoi)

radar_now   = get_radar(NOW_START,   NOW_END)
radar_flood = get_radar(FLOOD_START, FLOOD_END)
radar_dry   = get_radar(DRY_START,   DRY_END)

print("✓ Radar images ready for all 3 periods")

✓ Radar images ready for all 3 periods


## Step 5 - Train a Random Forest to detect water 

In [ ]:
# Get the JRC permanent water dataset for auto-labelling
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence")

water_label = jrc.gt(80).rename("label") # Label water if its been water > 80% of the time historically
keep_pixels = jrc.gt(80).Or(jrc.eq(0))   # Sample from clear-cut pixels
labelled = water_label.updateMask(keep_pixels)

# Stack the labels onto the radar image
training_image = radar_dry.addBands(labelled)

print("✓ Labels ready (from JRC water dataset)")

✓ Labels ready (from JRC water dataset)


In [ ]:
# Sample 500 random pixels - stratified sampling
samples = training_image.stratifiedSample(
    numPoints=500,
    classBand="label",
    region=aoi,
    scale=10,
    seed=42
)

# Split into 70 / 30
samples = samples.randomColumn("rand", seed=42)
train_set = samples.filter(ee.Filter.lt("rand", 0.7))
test_set  = samples.filter(ee.Filter.gte("rand", 0.7))

print(f"Training samples: {train_set.size().getInfo()}")
print(f"Testing samples : {test_set.size().getInfo()}")

Training samples: 710
Testing samples : 290


In [ ]:
# Train the Random Forest
rf_model = ee.Classifier.smileRandomForest(numberOfTrees=50, seed=42).train(
    features=train_set,
    classProperty="label",
    inputProperties=["VV", "VH"]    # the radar bands are our features
)

# Test it on the held-out test set
test_predictions = test_set.classify(rf_model)
confusion = test_predictions.errorMatrix("label", "classification")

accuracy = confusion.accuracy().getInfo()
print(f"🎯 Model accuracy: {accuracy*100:.1f}%")
print(f"   Confusion matrix: {confusion.getInfo()}")
print("   (rows = actual, columns = predicted; diagonal = correct)")

🎯 Model accuracy: 100.0%
   Confusion matrix: [[158, 0], [0, 132]]
   (rows = actual, columns = predicted; diagonal = correct)


## Step 6 - Apply the model + find the flood

In [14]:
# Classify both images using trained model
water_today = radar_now.classify(rf_model).rename("water_today")
water_dry   = radar_dry.classify(rf_model).rename("water_dry")

# Flood = water today AND NOT water in dry season
flood = water_today.And(water_dry.Not()).rename("flood")

# Clean up noisy pixels
flood = flood.focalMode(radius=1, kernelType="square", iterations=1)

print("✓ Flood map computed")


✓ Flood map computed


## Step 7 - Find the rice fields (paddy)

In [ ]:
# Use ESA WorldCover to mask out non-cropland areas
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().clip(aoi)
paddy = worldcover.eq(40).rename("paddy")    # Class 40 = cropland

print("✓ Paddy map ready (from ESA WorldCover)")

✓ Paddy map ready (from ESA WorldCover)


## Step 8 - Overlay: how much paddy is flooded?

In [ ]:
flooded_paddy = flood.And(paddy).rename("flooded_paddy")

# Compute areas in hectares
pixel_area = ee.Image.pixelArea()

def area_hectares(mask, name):
    total = (mask.multiply(pixel_area)
                 .reduceRegion(ee.Reducer.sum(), aoi, scale=10, maxPixels=1e10)
                 .getInfo())
    ha = list(total.values())[0] / 10_000
    print(f"  {name:<25s} {ha:>10,.1f} ha")
    return ha

print("\n📊 Results:")
paddy_ha   = area_hectares(paddy,         "Total paddy fields")
flood_ha   = area_hectares(flood,         "Total flood area")
damage_ha  = area_hectares(flooded_paddy, "Flooded paddy (damage)")

if paddy_ha > 0:
    percent = damage_ha / paddy_ha * 100
    print(f"\n {percent:.1f}% of paddy fields in this area are flooded")


📊 Results:
  Total paddy fields           3,964.0 ha
  Total flood area                12.9 ha
  Flooded paddy (damage)           0.2 ha

⚠️  0.0% of paddy fields in this area are flooded


## Step 9 - Show interactive map

In [ ]:
m = geemap.Map(center=[lat, lon], zoom=12)
m.add_basemap("HYBRID")

# Background: today's radar
m.addLayer(radar_now, {"min": -25, "max": 0, "bands": ["VV"]}, "Radar today")

# Layers
m.addLayer(water_dry.selfMask(),    {"palette": ["#0033cc"]}, "Permanent water")
m.addLayer(paddy.selfMask(),        {"palette": ["#ffcc00"]}, "Paddy fields")
m.addLayer(flood.selfMask(),        {"palette": ["#00ccff"]}, "Flood")
m.addLayer(flooded_paddy.selfMask(),{"palette": ["#ff0033"]}, "⚠️ Flooded paddy")

m.centerObject(aoi, 12)
m

Map(center=[3.506436282557175, 101.1027489105285], controls=(WidgetControl(options=['position', 'transparent_b…

## Step 10 — Export results

Save the flood-on-paddy polygons as a GeoJSON file. You can open this in QGIS, share with stakeholders, or load it into a web map (Leaflet, Mapbox).

In [18]:
# Turn the raster mask into vector polygons
polygons = (flooded_paddy.selfMask()
            .reduceToVectors(geometry=aoi,
                             scale=10,
                             geometryType="polygon",
                             eightConnected=True,
                             labelProperty="flooded",
                             maxPixels=1e10))

geojson = polygons.getInfo()
output_name = f"flooded_paddy_{today}.geojson"
with open(output_name, "w") as f:
    json.dump(geojson, f)

print(f"💾 Saved {len(geojson['features'])} polygons to {output_name}")
print(f"   You can open this file in QGIS or load it on a web map")


💾 Saved 7 polygons to flooded_paddy_2026-05-25.geojson
   You can open this file in QGIS or load it on a web map
